# Business Report Generator with LangGraph

LangGraph 기반 비즈니스 보고서 자동 생성 워크플로우

## Features
- 사용자 입력(필수 + 선택) 기반 Table of Contents 자동 생성
- 각 페이지 콘텐츠 **병렬 생성** (A4 사이즈 적합)
- 마크다운 형식 출력
- Gemini / Azure OpenAI 지원

## Cell 1: Setup & Imports

In [22]:
# Install dependencies (uncomment if needed)
# !pip install langgraph langchain-core langchain-google-genai langchain-openai python-dotenv pydantic

In [23]:
# Standard library
from typing import Annotated, Literal
from typing_extensions import TypedDict
import operator
import os
import json

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# LangChain LLMs
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import AzureChatOpenAI

# Utilities
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# Load environment variables
load_dotenv()

print("Imports loaded successfully!")

Imports loaded successfully!


## Cell 2: Input & State Schema Definition

In [24]:
# =============================================================================
# Input Schema - 사용자 입력 구조
# =============================================================================

class ReportInput(TypedDict, total=False):
    """사용자 입력 스키마 (필수 + 선택 항목)"""
    
    # 필수 입력
    report_type: str       # 주간업무보고, 기획안, 프로젝트 제안서, 결과 보고서, 회의록 등
    purpose: str           # 승인 요청, 현황 공유, 의사결정 요청, 아이디어 제안 등
    audience: str          # 직속 상사, 임원, 타부서, 외부 클라이언트 등
    topic: str             # 주제/제목
    key_message: str       # 핵심 메시지나 키워드
    company_info: str      # 회사/팀 이름, 업종
    
    # 선택 입력 (퀄리티 향상용)
    tone: str              # 격식체/반말, 간결함/상세함
    page_count: int        # 목표 페이지 수 (default: 3)
    emphasis: str          # 강조하고 싶은 부분
    include_visuals: bool  # 시각화 요소 포함 여부 (차트, 표 설명 등)
    additional_data: str   # 관련 데이터나 자료 (텍스트)


# =============================================================================
# Page Content Schema
# =============================================================================

class PageContent(TypedDict):
    """개별 페이지 콘텐츠 구조"""
    page_id: str       # 고유 식별자 (예: "exec_summary", "analysis")
    title: str         # 섹션 제목
    content: str       # A4 적합 마크다운 콘텐츠 (800-1200자)
    order: int         # 정렬 순서


# =============================================================================
# Workflow State Schema
# =============================================================================

class ReportState(TypedDict):
    """워크플로우 상태 스키마 (Reducer 포함)"""
    
    # 입력
    input: ReportInput
    
    # 생성된 목차 (TOC)
    toc: list[dict]  # [{page_id, title, description, order}, ...]
    
    # 페이지 콘텐츠 - 병렬 처리를 위해 Reducer 사용 필수!
    pages: Annotated[list[PageContent], operator.add]
    
    # 최종 출력
    final_report: str
    
    # 상태 메타데이터
    status: str


print("State schemas defined!")
print(f"- ReportInput: 필수 6개 + 선택 5개 필드")
print(f"- PageContent: page_id, title, content, order")
print(f"- ReportState: input, toc, pages (reducer), final_report, status")

State schemas defined!
- ReportInput: 필수 6개 + 선택 5개 필드
- PageContent: page_id, title, content, order
- ReportState: input, toc, pages (reducer), final_report, status


## Cell 3: LLM Configuration

In [25]:
# =============================================================================
# LLM Provider Configuration
# =============================================================================

def get_gemini_llm(temperature: float = 0.7) -> ChatGoogleGenerativeAI:
    """Google Gemini LLM 초기화"""
    return ChatGoogleGenerativeAI(
        model=os.getenv("GEMINI_MODEL", "gemini-2.0-flash"),
        google_api_key=os.getenv("GOOGLE_API_KEY"),
        temperature=temperature
    )


def get_azure_llm(temperature: float = 0.7) -> AzureChatOpenAI:
    """Azure OpenAI LLM 초기화"""
    return AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("OPENAI_API_VERSION", "2024-08-01-preview"),
        deployment_name="gpt-4o",
        temperature=temperature
    )


def get_llm(provider: Literal["gemini", "azure"] = "gemini", temperature: float = 0.7):
    """
    LLM Factory Function - 런타임에 프로바이더 선택 가능
    
    Args:
        provider: "gemini" 또는 "azure"
        temperature: 생성 온도 (0.0-1.0)
    
    Returns:
        LLM instance
    """
    if provider == "gemini":
        return get_gemini_llm(temperature)
    elif provider == "azure":
        return get_azure_llm(temperature)
    else:
        raise ValueError(f"Unknown provider: {provider}. Use 'gemini' or 'azure'.")


# Default LLM provider setting
LLM_PROVIDER: Literal["gemini", "azure"] = "gemini"

print(f"LLM Configuration ready!")
print(f"- Default provider: {LLM_PROVIDER}")
print(f"- Gemini model: {os.getenv('GEMINI_MODEL', 'gemini-2.0-flash')}")
print(f"- Azure endpoint configured: {bool(os.getenv('AZURE_OPENAI_ENDPOINT'))}")

LLM Configuration ready!
- Default provider: gemini
- Gemini model: gemini-3-pro-preview
- Azure endpoint configured: True


## Cell 4: TOC Generation Node

In [26]:
# =============================================================================
# Node 1: Table of Contents (TOC) Generation
# =============================================================================

def extract_text_content(response_content) -> str:
    """
    LLM 응답에서 텍스트 콘텐츠를 추출합니다.
    일부 LLM(Gemini 등)은 content를 리스트로 반환할 수 있습니다.
    """
    if isinstance(response_content, str):
        return response_content
    elif isinstance(response_content, list):
        # 리스트인 경우 텍스트 부분만 추출
        text_parts = []
        for item in response_content:
            if isinstance(item, str):
                text_parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                text_parts.append(item["text"])
            elif hasattr(item, "text"):
                text_parts.append(item.text)
        return "".join(text_parts)
    else:
        return str(response_content)


def generate_toc(state: ReportState) -> dict:
    """
    보고서 목차(TOC) 생성 노드
    
    사용자 입력(report_type, purpose, audience 등)을 기반으로
    보고서에 적합한 섹션 구조를 동적으로 생성합니다.
    
    Returns:
        dict: {"toc": [...], "status": "toc_generated"}
    """
    llm = get_llm(LLM_PROVIDER, temperature=0.5)  # 구조화된 출력을 위해 낮은 temperature
    user_input = state["input"]
    
    # 선택 입력 처리
    page_count = user_input.get("page_count", 3)
    tone = user_input.get("tone", "격식체, 간결함")
    emphasis = user_input.get("emphasis", "")
    include_visuals = user_input.get("include_visuals", False)
    additional_data = user_input.get("additional_data", "")
    
    prompt = f"""
당신은 비즈니스 보고서 구조 전문가입니다.
다음 정보를 바탕으로 보고서의 목차(Table of Contents)를 생성하세요.

## 보고서 정보
- 보고서 유형: {user_input.get("report_type", "일반 보고서")}
- 보고 목적: {user_input.get("purpose", "현황 공유")}
- 보고 대상: {user_input.get("audience", "직속 상사")}
- 주제/제목: {user_input.get("topic", "제목 없음")}
- 핵심 메시지: {user_input.get("key_message", "")}
- 회사/팀 정보: {user_input.get("company_info", "")}
- 목표 페이지 수: {page_count}페이지
- 강조 포인트: {emphasis if emphasis else "없음"}
- 시각화 요소 포함: {"예" if include_visuals else "아니오"}
- 추가 데이터/자료: {additional_data if additional_data else "없음"}

## 요구사항
1. 보고서 유형과 목적에 맞는 적절한 섹션 구조를 생성하세요.
2. 각 섹션은 A4 1페이지 분량으로 작성될 예정입니다.
3. 보고 대상의 수준에 맞게 섹션을 구성하세요.
4. 목표 페이지 수({page_count})에 맞춰 섹션 수를 조절하세요.

## 출력 형식
반드시 아래 JSON 배열 형식으로만 출력하세요. 다른 텍스트는 포함하지 마세요.

[
    {{"page_id": "고유_영문_id", "title": "섹션 제목", "description": "이 섹션에서 다룰 내용 설명 (2-3문장)", "order": 1}},
    {{"page_id": "다음_id", "title": "다음 섹션 제목", "description": "설명...", "order": 2}}
]

JSON 배열만 출력하세요:
"""
    
    response = llm.invoke(prompt)
    
    # JSON 파싱
    try:
        # 텍스트 콘텐츠 추출 (리스트 응답 처리)
        content = extract_text_content(response.content)
        
        # Markdown 코드 블록 제거
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0]
        elif "```" in content:
            content = content.split("```")[1].split("```")[0]
        
        toc = json.loads(content.strip())
        print(f"TOC 생성 완료: {len(toc)}개 섹션")
        
    except (json.JSONDecodeError, IndexError, TypeError) as e:
        print(f"JSON 파싱 실패, 기본 TOC 사용: {e}")
        # 기본 TOC (fallback)
        toc = [
            {"page_id": "summary", "title": "요약", "description": "보고서 핵심 내용 요약", "order": 1},
            {"page_id": "main_content", "title": "본문", "description": "주요 내용 상세 기술", "order": 2},
            {"page_id": "conclusion", "title": "결론 및 제언", "description": "결론과 향후 계획", "order": 3}
        ]
    
    return {
        "toc": toc,
        "status": "toc_generated"
    }


print("TOC Generation Node defined!")

TOC Generation Node defined!


## Cell 5: Page Content Generation Node (Parallel)

In [27]:
# =============================================================================
# Node 2: Page Content Generation (병렬 실행)
# =============================================================================

def generate_page_content(state: dict) -> dict:
    """
    개별 페이지 콘텐츠 생성 노드 (병렬 실행됨)
    
    이 노드는 Send() 객체를 통해 호출되며, 
    각 페이지에 대해 병렬로 실행됩니다.
    
    Note: state는 Send payload이며, 전체 ReportState가 아닙니다.
    
    Args:
        state: {"page_info": dict, "user_input": ReportInput}
    
    Returns:
        dict: {"pages": [PageContent]}  # 리스트로 반환 (reducer용)
    """
    llm = get_llm(LLM_PROVIDER, temperature=0.7)
    
    page_info = state["page_info"]
    user_input = state["user_input"]
    
    # 선택 입력 처리
    tone = user_input.get("tone", "격식체, 간결함")
    include_visuals = user_input.get("include_visuals", False)
    additional_data = user_input.get("additional_data", "")
    
    visual_instruction = ""
    if include_visuals:
        visual_instruction = """
- 적절한 위치에 표(Table)나 차트 설명을 마크다운으로 포함하세요.
- 예: | 항목 | 값 | 또는 [차트: 분기별 매출 추이]
"""
    
    prompt = f"""
당신은 비즈니스 보고서 작성 전문가입니다.
A4 1페이지 분량의 보고서 섹션을 마크다운으로 작성하세요.

## 보고서 맥락
- 보고서 유형: {user_input.get("report_type", "일반 보고서")}
- 보고 목적: {user_input.get("purpose", "현황 공유")}
- 보고 대상: {user_input.get("audience", "직속 상사")}
- 전체 주제: {user_input.get("topic", "제목 없음")}
- 핵심 메시지: {user_input.get("key_message", "")}
- 회사/팀: {user_input.get("company_info", "")}

## 현재 섹션 정보
- 섹션 제목: {page_info["title"]}
- 섹션 설명: {page_info["description"]}
- 섹션 순서: {page_info["order"]}번째

## 작성 지침
- 문체/톤: {tone}
- 분량: A4 1페이지 (800-1200자, 한글 기준)
- 형식: 마크다운 (##, ###, -, **굵게**, *기울임* 활용)
- 구성: 소제목 2-3개로 나누어 작성{visual_instruction}
- 참고 데이터: {additional_data if additional_data else "없음"}

## 주의사항
- 섹션 제목(## {page_info["title"]})부터 시작하세요.
- 전문적이고 읽기 쉬운 비즈니스 문서 스타일로 작성하세요.
- 구체적인 내용을 포함하되, 없는 수치는 만들지 마세요.

마크다운 콘텐츠:
"""
    
    response = llm.invoke(prompt)
    
    # 텍스트 콘텐츠 추출 (리스트 응답 처리)
    content = extract_text_content(response.content)
    
    print(f"페이지 생성 완료: {page_info['title']} (order: {page_info['order']})")
    
    # 반드시 리스트로 반환 (operator.add reducer용)
    return {
        "pages": [{
            "page_id": page_info["page_id"],
            "title": page_info["title"],
            "content": content,
            "order": page_info["order"]
        }]
    }


print("Page Content Generation Node defined!")

Page Content Generation Node defined!


## Cell 6: Fan-out & Combine Nodes

In [28]:
# =============================================================================
# Fan-out Function: 병렬 작업 분배
# =============================================================================

def fan_out_to_pages(state: ReportState) -> list[Send]:
    """
    TOC의 각 항목에 대해 병렬 페이지 생성 작업을 분배합니다.
    
    이 함수는 conditional edge에서 호출되며,
    list[Send]를 반환하여 병렬 실행을 트리거합니다.
    
    Returns:
        list[Send]: 각 페이지에 대한 Send 객체 리스트
    """
    sends = []
    
    for page_info in state["toc"]:
        # Send 객체: (target_node, payload)
        sends.append(
            Send(
                "generate_page_content",  # 대상 노드
                {
                    "page_info": page_info,
                    "user_input": state["input"]
                }
            )
        )
    
    print(f"Fan-out: {len(sends)}개 페이지 병렬 생성 시작")
    return sends


# =============================================================================
# Node 3: Combine Report (결과 조합)
# =============================================================================

def combine_report(state: ReportState) -> dict:
    """
    병렬 생성된 모든 페이지를 조합하여 최종 보고서를 생성합니다.
    
    페이지들은 order 기준으로 정렬되어 마크다운 문서로 조립됩니다.
    
    Returns:
        dict: {"final_report": str, "status": "completed"}
    """
    user_input = state["input"]
    
    # order 기준 정렬
    sorted_pages = sorted(state["pages"], key=lambda p: p["order"])
    
    # 최종 보고서 조립
    report_sections = []
    
    # 보고서 헤더
    report_sections.append(f"# {user_input.get('topic', '비즈니스 보고서')}\n")
    report_sections.append(f"**보고서 유형**: {user_input.get('report_type', '일반')}\n")
    report_sections.append(f"**작성 대상**: {user_input.get('audience', '')}\n")
    report_sections.append(f"**작성 팀**: {user_input.get('company_info', '')}\n")
    report_sections.append("\n---\n")
    
    # 목차
    report_sections.append("## 목차\n")
    for i, page in enumerate(sorted_pages, 1):
        report_sections.append(f"{i}. {page['title']}\n")
    report_sections.append("\n---\n")
    
    # 각 섹션 콘텐츠
    for page in sorted_pages:
        report_sections.append(f"\n{page['content']}\n")
        report_sections.append("\n---\n")
    
    final_report = "\n".join(report_sections)
    
    print(f"보고서 조합 완료: {len(sorted_pages)}개 섹션, {len(final_report)}자")
    
    return {
        "final_report": final_report,
        "status": "completed"
    }


print("Fan-out and Combine Nodes defined!")

Fan-out and Combine Nodes defined!


## Cell 7: Graph Construction

In [29]:
# =============================================================================
# LangGraph Workflow Construction
# =============================================================================

def create_report_generator():
    """
    보고서 생성 워크플로우 그래프를 구성하고 컴파일합니다.
    
    Workflow:
        START -> generate_toc -> [fan_out] -> generate_page_content (×N) -> combine_report -> END
    
    Returns:
        CompiledGraph: 실행 가능한 그래프
    """
    # StateGraph 초기화
    builder = StateGraph(ReportState)
    
    # 노드 추가
    builder.add_node("generate_toc", generate_toc)
    builder.add_node("generate_page_content", generate_page_content)
    builder.add_node("combine_report", combine_report)
    
    # 엣지 연결
    # 1. START -> TOC 생성
    builder.add_edge(START, "generate_toc")
    
    # 2. TOC -> Fan-out (병렬 페이지 생성)
    # fan_out_to_pages가 list[Send]를 반환하여 병렬 실행
    builder.add_conditional_edges(
        "generate_toc",
        fan_out_to_pages,
        ["generate_page_content"]  # 모든 Send의 대상 노드
    )
    
    # 3. 모든 페이지 생성 완료 -> Combine (자동 fan-in)
    builder.add_edge("generate_page_content", "combine_report")
    
    # 4. Combine -> END
    builder.add_edge("combine_report", END)
    
    # 그래프 컴파일
    graph = builder.compile()
    
    print("Report Generator Graph compiled!")
    print("Workflow: START -> generate_toc -> [parallel] generate_page_content -> combine_report -> END")
    
    return graph


# 그래프 생성
report_generator = create_report_generator()

Report Generator Graph compiled!
Workflow: START -> generate_toc -> [parallel] generate_page_content -> combine_report -> END


In [31]:
# 그래프 시각화 (Mermaid 다이어그램)
try:
    print(report_generator.get_graph().draw_mermaid())
except Exception as e:
    print(f"Mermaid diagram generation skipped: {e}")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate_toc(generate_toc)
	generate_page_content(generate_page_content)
	combine_report(combine_report)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate_toc;
	generate_page_content --> combine_report;
	generate_toc -.-> generate_page_content;
	combine_report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Cell 8: Test Execution

In [32]:
# =============================================================================
# 샘플 테스트: 주간업무보고
# =============================================================================

# 테스트 입력 데이터
sample_input: ReportInput = {
    # 필수 입력
    "report_type": "주간업무보고",
    "purpose": "현황 공유",
    "audience": "직속 상사",
    "topic": "2024년 4분기 마케팅팀 주간업무보고",
    "key_message": "신규 캠페인 성과 공유 및 다음 주 계획",
    "company_info": "ABC 주식회사 마케팅팀",
    
    # 선택 입력
    "tone": "격식체, 간결함",
    "page_count": 3,
    "emphasis": "캠페인 ROI 성과",
    "include_visuals": True,
    "additional_data": "지난 주 디지털 광고 클릭률 15% 상승, 전환율 3.2% 달성"
}

print("Sample input configured:")
for key, value in sample_input.items():
    print(f"  - {key}: {value}")

Sample input configured:
  - report_type: 주간업무보고
  - purpose: 현황 공유
  - audience: 직속 상사
  - topic: 2024년 4분기 마케팅팀 주간업무보고
  - key_message: 신규 캠페인 성과 공유 및 다음 주 계획
  - company_info: ABC 주식회사 마케팅팀
  - tone: 격식체, 간결함
  - page_count: 3
  - emphasis: 캠페인 ROI 성과
  - include_visuals: True
  - additional_data: 지난 주 디지털 광고 클릭률 15% 상승, 전환율 3.2% 달성


In [33]:
# =============================================================================
# 워크플로우 실행
# =============================================================================

# 초기 상태 구성
initial_state: ReportState = {
    "input": sample_input,
    "toc": [],
    "pages": [],
    "final_report": "",
    "status": "pending"
}

print("Starting report generation...")
print("="*50)

# 그래프 실행
result = report_generator.invoke(initial_state)

print("="*50)
print(f"\nGeneration completed!")
print(f"Status: {result['status']}")
print(f"TOC sections: {len(result['toc'])}")
print(f"Pages generated: {len(result['pages'])}")
print(f"Report length: {len(result['final_report'])} characters")

Starting report generation...
TOC 생성 완료: 3개 섹션
Fan-out: 3개 페이지 병렬 생성 시작
페이지 생성 완료: 2. 캠페인 ROI 및 데이터 시각화 분석 (order: 2)
페이지 생성 완료: 1. 주간 핵심 성과 요약 (Weekly Highlights) (order: 1)
페이지 생성 완료: 3. 차주 업무 계획 및 이슈 사항 (order: 3)
보고서 조합 완료: 3개 섹션, 6141자

Generation completed!
Status: completed
TOC sections: 3
Pages generated: 3
Report length: 6141 characters


In [19]:
# =============================================================================
# 생성된 보고서 출력
# =============================================================================

from IPython.display import Markdown, display

print("Generated Report:")
print("="*50)
display(Markdown(result["final_report"]))

Generated Report:


# 2024년 4분기 마케팅팀 주간업무보고

**보고서 유형**: 주간업무보고

**작성 대상**: 직속 상사

**작성 팀**: ABC 주식회사 마케팅팀


---

## 목차

1. 1. 금주 핵심 성과 요약 (Weekly Highlights)

2. 2. 캠페인별 상세 성과 및 데이터 분석

3. 3. 차주 업무 계획 및 이슈 사항


---


## 1. 금주 핵심 성과 요약 (Weekly Highlights)

본 섹션은 2024년 4분기 마케팅팀의 주간 주요 업무 성과를 기술합니다. 금주는 신규 캠페인의 본격적인 운영 단계로, 초기 트래픽 확보를 넘어 **실질적인 전환(Conversion)과 투자 대비 수익률(ROI)** 최적화에 주력했습니다. 특히 디지털 광고 채널의 소재 최적화를 통해 전주 대비 유의미한 효율 개선을 달성했습니다.

### 1.1. 신규 캠페인 퍼포먼스 지표 (KPI) 달성 현황

금주 가장 괄목할 만한 성과는 신규 캠페인의 고객 반응률 급상승입니다. 타겟 오디언스 세분화 및 소재 A/B 테스트 결과를 실시간으로 반영한 결과, 주요 성과 지표가 전주 대비 큰 폭으로 개선되었습니다.

*   **클릭률 (CTR) 대폭 상승:** 메인 배너 및 영상 광고의 크리에이티브 교체를 통해 지난주 대비 **15% 상승**한 클릭률을 기록했습니다. 이는 고객의 초기 관여도가 높아졌음을 시사합니다.
*   **목표 전환율 (CVR) 조기 달성:** 유입된 트래픽의 랜딩 페이지 이탈률을 개선하여, 당초 4분기 중반 목표였던 **전환율 3.2%**를 금주에 조기 달성했습니다.

**[표 1: 주간 주요 성과 지표 비교]**

| 구분 | 지난주 (W-1) | **금주 (Current)** | 증감률 (WoW) | 비고 |
| :--- | :---: | :---: | :---: | :--- |
| **클릭률 (CTR)** | 기준치 | **+15% 상승** | ▲ 15.0% | 소재 최적화 효과 |
| **전환율 (CVR)** | 2.8% (추정) | **3.2%** | ▲ 0.4%p | 랜딩페이지 UX 개선 |
| **ROAS** | 유지 | **상승** | ▲ 상승세 | 고효율 매체 집중 |

> **Insight:** 단순 유입량 증가뿐만 아니라, 구매 및 문의로 이어지는 전환율이 동반 상승한 점은 캠페인의 타겟팅 정확도가 매우 높음을 의미합니다.

### 1.2. 채널별 ROI 최적화 및 예산 효율화

한정된 예산 내에서 최대의 성과를 내기 위해, 채널별 기여도 분석에 기반한 예산 재배치(Re-allocation)를 진행했습니다. 성과가 저조한 일부 디스플레이 네트워크의 비중을 축소하고, 검색 광고 및 리타겟팅 매체의 비중을 확대했습니다.

*   **저효율 지면 제외:** 클릭 당 비용(CPC)은 높으나 체류 시간이 짧은 3개 매체에 대한 송출을 중단하여 불필요한 예산 누수를 차단했습니다.
*   **고관여 타겟 집중:** 전환율 3.2% 달성을 견인한 고효율 키워드 그룹에 예산을 20% 증액 편성하여 ROAS(광고비 대비 매출액)를 극대화했습니다.

**[차트: 채널별 예산 비중 변화 및 예상 ROI 추이]**
*(참고: 지난주 대비 검색 광고 및 리타겟팅 비중 확대, 디스플레이 비중 축소 그래프)*

### 1.3. 주요 운영 하이라이트 및 개선 사항

정량적 수치 외에 캠페인 운영 측면에서도 다음과 같은 질적 개선이 이루어졌습니다.

1.  **크리에이티브 리프레시(Refresh):** 광고 피로도를 낮추기 위해 주 중반 신규 이미지 소재 2종을 추가 투입하였으며, 이는 후반부 CTR 하락 방어에 주효했습니다.
2.  **CS 및 피드백 반영:** 캠페인 런칭 초기 고객 문의 사항(FAQ)을 분석하여 상세 페이지 상단에 '자주 묻는 질문' 섹션을 보강, 고객의 구매 결정 시간을 단축시켰습니다.

---
**요약 (Summary):**
금주는 **CTR 15% 상승**과 **CVR 3.2% 달성**이라는 구체적인 수치를 통해 신규 캠페인의 시장 적합성을 확인한 한 주였습니다. 이러한 성과를 바탕으로 차주에는 리타겟팅 모수를 확대하여 매출 볼륨을 키우는 데 집중할 계획입니다.


---


## 2. 캠페인별 상세 성과 및 데이터 분석

본 섹션에서는 2024년 4분기 주력 캠페인의 지난 주차(W4) 디지털 광고 성과를 정량적으로 분석하고, 채널별 예산 집행 대비 효율(ROI)을 점검합니다. 특히 신규 소재 도입에 따른 지표 변화를 중점적으로 기술합니다.

### 2.1. 디지털 광고 성과 추이 (Key Metrics Trend)

지난 주 실시한 신규 크리에이티브 A/B 테스트 및 타겟팅 최적화 작업의 결과로, 주요 성과 지표가 전주 대비 뚜렷한 상승세를 보였습니다.

*   **주요 성과 요약**:
    *   **클릭률(CTR) 상승**: 직전 주차 대비 **15% 상승**하며 고객의 초기 반응률이 크게 개선되었습니다. 이는 시즌 이슈를 반영한 카피 변경과 고해상도 이미지 소재 교체가 주효했던 것으로 분석됩니다.
    *   **전환율(CVR) 달성**: 유입 품질 개선에 힘입어 목표치였던 3.0%를 상회하는 **3.2%의 전환율**을 달성했습니다. 이는 4분기 캠페인 런칭 이후 최고 수치입니다.

**[차트 2-1: 주간 CTR 및 CVR 성장 추이]**
*(설명: X축은 지난 4주간의 기간, Y축은 퍼센트(%)를 나타내며, 막대그래프는 CTR, 꺾은선 그래프는 CVR을 표시함. W4 지점에서 두 지표 모두 우상향하는 추세를 시각화)*

| 구분 | 전주 대비 증감 | 금주 성과 | 비고 |
| :--- | :---: | :---: | :--- |
| **CTR (클릭률)** | ▲ 15% | **상승** | 신규 소재(B안) 반응 호조 |
| **CVR (전환율)** | - | **3.2%** | 목표(3.0%) 초과 달성 |
| **CPA (전환당비용)** | ▼ 감소 | **개선** | 효율 최적화 성공 |

### 2.2. 채널별 예산 집행 현황 및 ROI 비교

4분기 마케팅 예산 계획에 따라 주요 매체별로 예산을 집행하였으며, 성과 데이터에 기반하여 채널별 기여도를 평가했습니다.

*   **예산 집행 효율성**: 전체 예산은 계획 범위 내에서 정상 집행되었으며, 성과가 저조한 일부 디스플레이 네트워크(GDN)의 예산을 축소하고, 전환 효율이 높은 소셜 미디어 채널로 **Re-allocation(재배정)** 하였습니다.
*   **채널별 ROI 분석**:
    *   **Meta (Instagram/Facebook)**: 신규 숏폼 영상 소재가 높은 인게이지먼트를 유도하며 가장 높은 ROI를 기록했습니다. 상기 언급된 CTR 15% 상승의 주요 견인 채널입니다.
    *   **Search Ads (SA)**: 브랜드 키워드 검색량은 안정적이나, 일반 키워드의 경쟁 심화로 클릭당 비용(CPC)이 소폭 상승했습니다. 단, 전환율(3.2%) 기여도는 여전히 최상위를 유지 중입니다.

**[표 2-1: 채널별 예산 집행 및 ROAS 효율표]**

| 채널명 | 예산 집행률 | ROAS (광고비 대비 매출) | 효율 등급 | 분석 코멘트 |
| :--- | :---: | :---: | :---: | :--- |
| **Meta** | 105% (증액) | **High** | **S** | 고효율 소재 발굴로 예산 증액 |
| **Google (SA)** | 100% | **High** | A | 고전환 키워드 중심 운영 유지 |
| **Youtube** | 95% | Medium | B | 브랜딩 목적 달성, 전환 기여는 보조적 |
| **Network 배너** | 80% (감액) | Low | C | 저효율 지면 제외 및 타겟팅 재설정 필요 |

### 2.3. 시사점 및 차주 최적화 전략

금주 데이터 분석을 통해 확인된 **'고효율 소재의 파급력'**과 **'타겟팅 정교화의 중요성'**을 바탕으로 차주 전략을 수립합니다.

1.  **Winning Creative 확장**: CTR 상승을 견인한 'B안(시즌 소구형)' 소재를 메인으로 교체하고, 해당 소재의 베리에이션(Variation) 버전을 3종 추가 제작하여 피로도를 낮추고 성과를 유지합니다.
2.  **전환 중심의 예산 운용**: 현재 **3.2%의 전환율**을 3.5%까지 끌어올리기 위해, 이탈률이 높은 랜딩페이지 구간의 UI/UX를 긴급 점검하고 개선합니다.
3.  **저효율 채널 구조조정**: ROAS 효율이 C등급인 네트워크 배너 광고의 비중을 10% 추가 축소하고, 확보된 예산을 퍼포먼스가 검증된 Meta 채널의 리타겟팅 캠페인에 투입하여 구매 전환을 가속화할 계획입니다.


---


## 3. 차주 업무 계획 및 이슈 사항

금주 달성한 **디지털 광고 클릭률(CTR) 15% 상승** 및 **전환율(CVR) 3.2%**라는 고무적인 성과를 바탕으로, 차주(W42)는 유입된 트래픽을 실제 매출로 연결하는 '전환 최적화'와 '캠페인 효율 극대화'에 역량을 집중할 계획입니다. 이를 위한 세부 추진 과제와 일정, 경영진의 의사결정이 필요한 항목은 다음과 같습니다.

### 3.1. 중점 추진 과제: 성과 기반 캠페인 고도화

지난주 데이터 분석 결과, 고효율 소재와 저효율 소재 간의 성과 격차가 뚜렷하게 확인되었습니다. 이에 따라 차주에는 예산의 효율적 재배분과 타겟 정교화 작업을 최우선으로 진행합니다.

*   **고효율 소재 예산 증액 및 확산**
    *   성과가 검증된 A/B안(CTR 15% 상회 그룹)에 대해 일일 예산을 20% 증액하여 노출 점유율을 확대합니다.
    *   전환율 3.2%를 기록한 랜딩 페이지의 UI/UX 요소를 타 채널(SNS, 디스플레이 광고) 소재에도 일관되게 적용하여 브랜드 경험을 통일합니다.

*   **이탈률 방어 및 리타겟팅 강화**
    *   유입 후 구매 없이 이탈한 고객(약 96.8%)을 대상으로 개인화된 혜택(첫 구매 쿠폰 등)을 노출하는 리타겟팅 광고를 집행합니다.
    *   [차트: 퍼널별 고객 이탈 구간 분석]을 기반으로, 장바구니 단계에서의 이탈을 줄이기 위한 CRM 메시지(알림톡, 앱 푸시) 발송 시나리오를 가동합니다.

*   **신규 채널 테스트 착수**
    *   기존 메타(Meta) 및 구글 광고 외에 2030 타겟 비중이 높은 숏폼 플랫폼(틱톡/릴스) 전용 소재 2종을 시범 운영하여 채널 확장을 모색합니다.

### 3.2. 주간 상세 업무 일정 (W42)

효율적인 캠페인 운영을 위해 요일별로 분석, 실행, 모니터링 단계를 구분하여 업무를 수행합니다.

| 요일 | 구분 | 주요 업무 내용 | 비고 |
| :--- | :--- | :--- | :--- |
| **월** | **성과 분석** | - 주말 간 광고 성과 데이터 취합 및 ROAS 분석<br>- 저효율 소재 OFF 및 입찰가 조정 | 데이터팀 협업 |
| **화** | **기획/제작** | - 리타겟팅용 신규 배너 소재(3종) 카피라이팅 및 디자인<br>- 숏폼 영상 편집 및 썸네일 제작 | 디자인팀 요청 |
| **수** | **광고 집행** | - 신규 소재 세팅 및 광고 검수 요청<br>- CRM 메시지(장바구니 리마인더) 발송 | |
| **목** | **모니터링** | - 신규 소재 초기 반응율(CTR, CPC) 점검<br>- 경쟁사 프로모션 현황 파악 및 대응 전략 수립 | |
| **금** | **보고/마감** | - 주간 성과 지표(KPI) 결산 및 차주 전략 수립<br>- W42 주간업무보고서 작성 | |

### 3.3. 의사결정 필요 사항 및 협조 요청

원활한 캠페인 최적화 및 신규 시도를 위해 아래 두 가지 사항에 대한 팀장님의 검토 및 유관 부서 협조를 요청드립니다.

1.  **잔여 예산의 채널 간 전용(Reallocation) 승인**
    *   **현황:** 검색광고(SA) 예산 일부가 키워드 경쟁 심화로 소진율이 저조함(계획 대비 80% 수준).
    *   **요청:** 미소진된 검색광고 예산 약 500만 원을 효율이 급상승 중인 **디스플레이 광고(DA) 예산으로 전용**하여 전환 모멘텀을 강화하고자 함.
    *   **기대효과:** 유입 단가(CPC) 10% 절감 및 전환 건수 약 50건 추가 확보 예상.

2.  **디자인팀 리소스 우선 배정 협조**
    *   **이슈:** 숏폼 플랫폼 테스트를 위한 영상 편집 리소스가 타 부서 요청 건으로 인해 지연되고 있음.
    *   **요청:** 4분기 주력 캠페인의 적기 런칭을 위해 **W42 화요일까지 영상 소재 2종 제작**이 우선순위로 배정될 수 있도록 디자인팀장님께 협조 요청 요망.

---
*작성자: 마케팅팀 김대리*
*작성일: 2024. 10. XX.*


---


## Cell 9: Interactive Demo (Optional)

In [20]:
# =============================================================================
# 재사용 가능한 보고서 생성 함수
# =============================================================================

def generate_report(
    report_type: str,
    purpose: str,
    audience: str,
    topic: str,
    key_message: str,
    company_info: str,
    tone: str = "격식체, 간결함",
    page_count: int = 3,
    emphasis: str = "",
    include_visuals: bool = False,
    additional_data: str = "",
    provider: Literal["gemini", "azure"] = "gemini"
) -> str:
    """
    비즈니스 보고서를 생성합니다.
    
    Args:
        report_type: 보고서 유형 (주간업무보고, 기획안, 제안서 등)
        purpose: 보고 목적 (승인요청, 현황공유, 의사결정 등)
        audience: 보고 대상 (직속상사, 임원, 외부 등)
        topic: 주제/제목
        key_message: 핵심 메시지
        company_info: 회사/팀 정보
        tone: 문체 (선택)
        page_count: 목표 페이지 수 (선택)
        emphasis: 강조 포인트 (선택)
        include_visuals: 시각화 요소 포함 여부 (선택)
        additional_data: 추가 데이터 (선택)
        provider: LLM 프로바이더 (선택)
    
    Returns:
        str: 마크다운 형식의 보고서
    """
    global LLM_PROVIDER
    LLM_PROVIDER = provider
    
    user_input: ReportInput = {
        "report_type": report_type,
        "purpose": purpose,
        "audience": audience,
        "topic": topic,
        "key_message": key_message,
        "company_info": company_info,
        "tone": tone,
        "page_count": page_count,
        "emphasis": emphasis,
        "include_visuals": include_visuals,
        "additional_data": additional_data
    }
    
    initial_state: ReportState = {
        "input": user_input,
        "toc": [],
        "pages": [],
        "final_report": "",
        "status": "pending"
    }
    
    result = report_generator.invoke(initial_state)
    
    return result["final_report"]


print("generate_report() function ready!")
print("\nUsage example:")
print('report = generate_report(')
print('    report_type="기획안",')
print('    purpose="승인 요청",')
print('    audience="임원",')
print('    topic="2025년 신규 서비스 런칭 기획안",')
print('    key_message="시장 선점을 위한 빠른 출시 필요",')
print('    company_info="XYZ 기업 사업개발팀"')
print(')')

generate_report() function ready!

Usage example:
report = generate_report(
    report_type="기획안",
    purpose="승인 요청",
    audience="임원",
    topic="2025년 신규 서비스 런칭 기획안",
    key_message="시장 선점을 위한 빠른 출시 필요",
    company_info="XYZ 기업 사업개발팀"
)


In [21]:
# =============================================================================
# 추가 테스트: 기획안 생성
# =============================================================================

# Uncomment to run
# proposal_report = generate_report(
#     report_type="기획안",
#     purpose="승인 요청",
#     audience="임원",
#     topic="2025년 AI 기반 고객 서비스 자동화 기획안",
#     key_message="고객 응대 시간 50% 단축, 만족도 향상",
#     company_info="테크스타트업 고객경험팀",
#     page_count=4,
#     include_visuals=True,
#     additional_data="현재 평균 응대 시간 15분, 목표 7분"
# )
# 
# display(Markdown(proposal_report))